# Session 4 · Connecting the physical world: MQTT, CoAP, LoRaWAN
**Blockchain and IoT · UNIE · Fourth-year Computer Engineering · Wednesday, September 23, 2026**

> **Question of the day:** *You have thirty seconds of airtime per device per day. What do you let it say?*

This notebook runs **alongside the presentation**. When **▶ COLAB · STEP N** appears on screen, come here and run that step.

**How to use it**
- Run the cells **in order**, from top to bottom (▶ to the left of each cell, or `Shift + Enter`).
- The **🧪 YOUR TURN** cells are yours: edit them, break them, and run them again. They do not affect the rest.
- If something gets stuck: *Runtime → Restart session*, then run again from **Step 0**.

| Step | What you do | Slide |
|---|---|---|
| 0 | Set up the environment (an MQTT broker in your Colab instance) | 7 |
| 1 | How large is one reading? JSON versus bytes | 30 |
| 2 | MQTT: publish, subscribe, and inspect the wire | 48 |
| 3 | Classroom MQTT: everyone on the same broker | 50 |
| 4 | CoAP: the same reading, connectionless, with a 4-byte header | 63 |
| 5 | LoRaWAN: how long each message stays on air | 80 |
| 6 | **What do you let it say?** Six strategies within the budget | 88 |
| 7 | The weak point: when the gateway that heard you disappears | 92 |
| 8 | Column 6 of your worksheet (ACT 1) | 104 |


---
## Step 0 · Setup · slide 7
Install an **MQTT broker** (Mosquitto, widely used worldwide) in your Colab instance and the session's two libraries: `paho-mqtt` (MQTT client) and `aiocoap` (CoAP). This takes about 40 seconds. **Run the cell and keep listening.**

In [ ]:
#@title ▶ Step 0 · Run and keep listening
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y mosquitto > /dev/null 2>&1
!pip -q install "paho-mqtt>=2.0" aiocoap > /dev/null 2>&1
!pgrep -x mosquitto > /dev/null || mosquitto -d
!sleep 1; pgrep -x mosquitto > /dev/null && echo "✅ MQTT broker running at localhost:1883" || echo "❌ Broker failed to start: restart the runtime and try again"

In [ ]:
#@title Your alias (not your real name: anyone can read what you publish in Step 3)
ALIAS = "reefer-017"  #@param {type:"string"}
ALIAS = ALIAS.strip().lower().replace(" ", "-") or "anonimo"

import json, struct, time, math, threading, socket, asyncio, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import paho.mqtt.client as mqtt
plt.rcParams.update({"figure.figsize": (11, 4.5), "font.size": 13, "axes.spines.top": False, "axes.spines.right": False})
random.seed(4); np.random.seed(4)
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Audio, display, Markdown
from matplotlib.patches import FancyBboxPatch

def diagrama(nodos, flechas, titulo="", ancho=11, alto=4):
    # nodes: {name: (x, y, color)} · arrows: [(source, target, label, color)]
    fig, ax = plt.subplots(figsize=(ancho, alto)); ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis("off")
    for n, (x, y, col) in nodos.items():
        ax.add_patch(FancyBboxPatch((x-0.95, y-0.38), 1.9, 0.76, boxstyle="round,pad=0.05", fc=col, ec="none", alpha=0.9))
        ax.text(x, y, n, ha="center", va="center", fontsize=12, color="white", weight="bold")
    for a, b, et, col in flechas:
        (x1, y1, _), (x2, y2, _) = nodos[a], nodos[b]
        ax.annotate("", xy=(x2 - 0.98*np.sign(x2-x1) if x2 != x1 else x2, y2 - 0.42*np.sign(y2-y1)),
                    xytext=(x1 + 0.98*np.sign(x2-x1) if x2 != x1 else x1, y1 + 0.42*np.sign(y2-y1)),
                    arrowprops=dict(arrowstyle="-|>", lw=2.2, color=col))
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.2, et, ha="center", fontsize=11, color=col,
                bbox=dict(fc="white", ec="none", alpha=0.85))
    ax.set_title(titulo, fontsize=14, loc="left"); plt.show()

def secuencia(filas, izq="cliente", der="servidor", titulo="", ax=None):
    # filas: [(direction '→' o '←', etiqueta)] → diagrama de secuencia como el de las diapositivas
    solo = ax is None
    if solo: fig, ax = plt.subplots(figsize=(7, 0.55*len(filas) + 1.6))
    n = len(filas); ax.set_xlim(-0.3, 3.3); ax.set_ylim(-n - 0.6, 0.9); ax.axis("off")
    for x, t in ((0, izq), (3, der)):
        ax.text(x, 0.5, t, ha="center", fontsize=12, weight="bold"); ax.plot([x, x], [0.2, -n - 0.4], color="#999", lw=1.5)
    for i, (s, et) in enumerate(filas):
        y = -i - 0.4; a, b = (0, 3) if s == "→" else (3, 0)
        col = "#d1495b" if "PUBLISH" in et or "2.05" in et or "GET" in et else "#2e4057"
        ax.annotate("", xy=(b, y - 0.25), xytext=(a, y), arrowprops=dict(arrowstyle="-|>", lw=2, color=col))
        ax.text(1.5, y - 0.02, et, ha="center", fontsize=10.5, color=col, bbox=dict(fc="white", ec="none", alpha=0.9))
    ax.set_title(titulo, fontsize=12.5)
    if solo: plt.show()

print("Ready. Your alias:", ALIAS)

---
## Step 1 · How large is one reading? · slide 30
A reading from a refrigerated shipping container (the *reefer* from session 3). We encode it four ways and **count the bytes**. Remember: over radio, **every byte costs airtime**, and airtime costs battery life and consumes your allowance.

In [ ]:
lectura = {"dispositivo": "reefer-017", "temperatura": 4.37, "humedad": 82.1,
           "bateria": 3.61, "ts": "2026-09-23T10:15:00Z"}

f1 = json.dumps(lectura, indent=2).encode()                       # Pretty JSON (human-readable formatting)
f2 = json.dumps(lectura, separators=(",", ":")).encode()          # Compact JSON
f3 = b"4.37,82.1,3.61"                                            # CSV: numbers only, no field names
# Binary: temperature in hundredths (int16), humidity in half-percent units (uint8), battery as (V-2)*100 (uint8)
f4 = struct.pack(">hBB", round(4.37*100), round(82.1*2), round((3.61-2)*100))

formatos = {"Pretty JSON": f1, "Compact JSON": f2, "CSV": f3, "Binary": f4}
for nombre, b in formatos.items():
    print(f"{nombre:14s} {len(b):4d} bytes   {b[:60]!r}{'…' if len(b) > 60 else ''}")
print("\nBinary in hexadecimal:", f4.hex(" "))

**What the binary encoding omits:** field names (the receiver already knows them), the device ID (already carried in the network header), and the timestamp (added by the receiver). That last choice has a cost: **the time now marks arrival, not measurement**. As we said in session 3: *data arrives with a timestamp, not with proof that it is true*.

In [ ]:
# Decode: the receiver must know EXACTLY what each byte means
t, h, v = struct.unpack(">hBB", f4)
print(f"temperature = {t/100} °C   humidity = {h/2} %HR   battery = {v/100 + 2} V")
print("Did we lose anything? Original humidity 82.1 → decoded", h/2, "(you choose the resolution when encoding)")

pd.Series({k: len(v) for k, v in formatos.items()}).plot.barh(color=["#b0b0b0", "#b0b0b0", "#7a9cc6", "#d1495b"])
plt.xlabel("bytes"); plt.title("The same reading in four formats"); plt.gca().invert_yaxis(); plt.show()

### 🧪 YOUR TURN 1 · Encode the sensor from your worksheet (ACT 1)
Take **your** sensor from the session 3 worksheet. Decide which quantities to measure, what **resolution** you actually need, and how many bytes to use. `struct` format codes: `b`/`B` = 1 byte (signed/unsigned), `h`/`H` = 2 bytes, `i`/`I` = 4 bytes.

In [ ]:
# 🧪 YOUR TURN 1 — SHT40: refrigerated-container temperature.
# Hundredths of a degree preserve the 8 °C threshold; int16 supports
# from -327.68 to 327.67 °C. Humidity is not needed for this alert.
mi_lectura = {"temperatura_c": 4.37}
FORMATO = ">h"  # signed integer, network byte order: 2 bytes
ESCALAS = [100]
mis_bytes = struct.pack(FORMATO, round(mi_lectura["temperatura_c"] * ESCALAS[0]))
vuelta = struct.unpack(FORMATO, mis_bytes)[0] / ESCALAS[0]
print(f"{len(mis_bytes)} bytes: {mis_bytes.hex(' ')} → {vuelta:.2f} °C")
print("In JSON it would be", len(json.dumps(mi_lectura, separators=(",", ":")).encode("utf-8")), "bytes")
print("The receiver must know the scale and format; it timestamps the reading on arrival.")


---
## Step 2 · MQTT: publish, subscribe, and inspect the wire · slide 48

> ### 💬 MQTT in one sentence
> **It is like a WhatsApp group with channels.** The sensor posts to a channel (`reefer/017/temp`); an intermediate server, the **broker**, forwards the message to everyone subscribed to it. **The sensor does not know who reads it, and readers do not know where the sensor is.**

| Term | Meaning | WhatsApp analogy |
|---|---|---|
| **broker** | the server that receives and distributes messages | WhatsApp's servers |
| **topic** | the message address, separated by `/` | the group name |
| **publish** | send a message to a topic | post to the group |
| **subscribe** | request messages from a topic | join the group |
| **QoS** | the delivery assurance level | one tick, two ticks… |

In [ ]:
diagrama({"sensor 017": (1.3, 3, "#d1495b"), "sensor 018": (1.3, 1, "#d1495b"), "BROKER": (5, 2, "#2e4057"),
          "office dashboard": (8.7, 3, "#00798c"), "technician phone": (8.7, 1, "#00798c")},
         [("sensor 017", "BROKER", "reefer/017/temp", "#d1495b"), ("sensor 018", "BROKER", "reefer/018/temp", "#d1495b"),
          ("BROKER", "office dashboard", "reefer/#", "#00798c"), ("BROKER", "technician phone", "reefer/017/temp", "#00798c")],
         "MQTT: everyone talks to the broker, not directly to each other")

To see what actually travels, we put a **“wire”** between the clients and the broker: a small program that forwards everything and **counts every packet and byte**. Clients connect to port **1884** (the wire), which forwards to **1883** (the broker).

In [ ]:
#@title Tool · The wire (run it; you need not read the code)
TIPOS = {1:"CONNECT",2:"CONNACK",3:"PUBLISH",4:"PUBACK",5:"PUBREC",6:"PUBREL",7:"PUBCOMP",
         8:"SUBSCRIBE",9:"SUBACK",10:"UNSUBSCRIBE",11:"UNSUBACK",12:"PINGREQ",13:"PINGRESP",14:"DISCONNECT"}

class Cable:
    def __init__(self, escucha=1884, destino=("127.0.0.1", 1883)):
        self.destino, self.registro, self.lock, self.n = destino, [], threading.Lock(), 0
        s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", escucha)); s.listen(50)
        threading.Thread(target=self._aceptar, args=(s,), daemon=True).start()
    def _aceptar(self, s):
        while True:
            c, _ = s.accept(); b = socket.create_connection(self.destino)
            self.n += 1; ctx = {"quien": f"connection {self.n}"}
            threading.Thread(target=self._bombear, args=(c, b, "client → broker", ctx), daemon=True).start()
            threading.Thread(target=self._bombear, args=(b, c, "broker → client", ctx), daemon=True).start()
    def _bombear(self, src, dst, direction, ctx):
        buf = b""
        while True:
            try: data = src.recv(4096)
            except OSError: data = b""
            if not data:                       # one side disconnected: close both ends
                for x in (src, dst):
                    try: x.shutdown(socket.SHUT_RDWR)
                    except OSError: pass
                    try: x.close()
                    except OSError: pass
                return
            try: dst.sendall(data)
            except OSError: return
            buf += data
            while len(buf) >= 2:
                val, mult, i, ok = 0, 1, 1, False
                while i < len(buf) and i < 5:
                    val += (buf[i] & 127) * mult; mult *= 128; i += 1
                    if not buf[i-1] & 128: ok = True; break
                if not ok or len(buf) < i + val: break
                pkt, buf = buf[:i+val], buf[i+val:]
                tipo = TIPOS.get(pkt[0] >> 4, "?")
                if tipo == "CONNECT":                     # leemos el client-id para saber who es
                    try:
                        L = int.from_bytes(pkt[i+10:i+12], "big"); ctx["quien"] = pkt[i+12:i+12+L].decode()
                    except Exception: pass
                with self.lock:
                    self.registro.append((time.time(), ctx["quien"], direction, tipo, len(pkt)))
    def limpiar(self):
        with self.lock: self.registro.clear()
    def ver(self, titulo=""):
        time.sleep(0.6)
        with self.lock: df = pd.DataFrame(self.registro, columns=["t", "who", "direction", "packet", "bytes"])
        if df.empty: print("(the wire has not seen anything)"); return df
        df["t"] = (df["t"] - df["t"].min()).round(3)
        print(f"── {titulo}  ·  {len(df)} packets, {df['bytes'].sum()} bytes MQTT "
              f"(+ ≥40 TCP/IP bytes per packet that this wire cannot see)")
        display(df); return df
    def secuencia(self, quien, titulo="", ax=None):
        time.sleep(0.6)
        with self.lock: reg = [r for r in self.registro if r[1] == quien]
        secuencia([("→" if r[2].startswith("cliente") else "←", f"{r[3]} · {r[4]} B") for r in reg],
                  izq=quien, der="broker", titulo=titulo, ax=ax)

try:
    cable
except NameError:
    cable = Cable()
print("🔌 Wire installed: clients connect to localhost:1884")

def cliente(nombre, puerto=1884, keepalive=60, will=None):
    c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=nombre)
    if will: c.will_set(*will)
    c.connect("127.0.0.1", puerto, keepalive=keepalive); c.loop_start(); time.sleep(0.3)
    return c

recibidos = []
def al_recibir(c, userdata, msg):
    recibidos.append({"topic": msg.topic, "payload": msg.payload, "bytes": len(msg.payload),
                      "qos": msg.qos, "retained": bool(msg.retain)})

### 2a · One subscriber and one publisher
The **dashboard** (a computer in the office) subscribes to everything under `reefer/`. The **sensor** publishes its temperature in 2 bytes.

In [ ]:
cable.limpiar(); recibidos.clear()
panel = cliente("panel-oficina");  panel.on_message = al_recibir
panel.subscribe("reefer/#", qos=0)
sensor = cliente("sensor-017")

sensor.publish("reefer/017/temp", struct.pack(">h", 437), qos=0).wait_for_publish(5)
sensor.publish("reefer/017/temp_json", json.dumps({"temperatura": 4.37}), qos=0).wait_for_publish(5)
time.sleep(0.5)
print("The dashboard received:", [(m["topic"], m["payload"]) for m in recibidos])
_ = cable.ver("Connect, subscribe, and publish twice")
cable.secuencia("sensor-017", "What the sensor did, packet by packet")

**Notice:** before sending a reading, the sensor has already exchanged `CONNECT` and `CONNACK`. Every `PUBLISH` also contains **the entire topic name**: the text `reefer/017/temp` weighs more than the reading.

### 2b · QoS: how many acknowledgments?
QoS 0 = *at most once*: send and forget (**one tick**). QoS 1 = *at least once*: the broker acknowledges receipt; without an acknowledgment the sender retries (**duplicates are possible**). QoS 2 = *exactly once*: a four-step handshake.

**🎲 Before running, guess:** how many packets does the sensor exchange to send 2 bytes with QoS 2? Enter your guess here 👉 `mi_apuesta = ?`

In [ ]:
mi_apuesta = 2   # ← change your guess and run
filas = []
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.8))
for q, ax in zip((0, 1, 2), ejes):
    cable.limpiar()
    sensor.publish("reefer/017/temp", struct.pack(">h", 437), qos=q).wait_for_publish(5)
    time.sleep(0.6)
    with cable.lock: reg = [r for r in cable.registro if r[1] == "sensor-017"]
    filas.append({"QoS": q, "sensor packets": " + ".join(r[3] for r in reg),
                  "packet count": len(reg), "bytes MQTT": sum(r[4] for r in reg)})
    cable.secuencia("sensor-017", f"QoS {q}", ax=ax)
plt.tight_layout(); plt.show()
print(f"Your guess: {mi_apuesta} · Actual QoS 2 result: {filas[2]['packet count']} packets")
pd.DataFrame(filas)

**For a 2-byte reading**, QoS 2 exchanges four packets. That matters little over Wi-Fi, but on battery-powered radio each packet wakes the radio. Also, “exactly once” applies **between sensor and broker**, not all the way to your database.

### 2c · Topics and wildcards
`+` = any one level. `#` = everything below this level. The broker determines who receives each message **from the topic name alone**.

In [ ]:
topics = ["reefer/017/temp", "reefer/017/humedad", "reefer/018/temp", "reefer/018/puerta/estado", "camion/9/gps"]
filtros = ["reefer/017/temp", "reefer/+/temp", "reefer/017/#", "reefer/#", "#"]
pd.DataFrame({f: ["✔" if mqtt.topic_matches_sub(f, t) else "" for t in topics] for f in filtros}, index=topics)

### 2d · Retained messages: late subscribers still get the latest value
A message sent with `retain=True` remains on the broker: **a subscriber joining later immediately receives the latest value**, without waiting for the next transmission.

In [ ]:
recibidos.clear()
sensor.publish("reefer/017/consigna", b"4.0", qos=1, retain=True).wait_for_publish(5)
time.sleep(0.3)
del_movil = []
movil = cliente("movil-tecnico")
movil.on_message = lambda c, u, m: del_movil.append((m.topic, m.payload, "retained" if m.retain else "live"))
movil.subscribe("reefer/017/consigna"); time.sleep(0.8)
print("The phone connected AFTER publication and still received:", del_movil)
movil.disconnect(); _ = movil.loop_stop()

### 2e · Last Will: the broker announces an unexpected disconnect
When connecting, the sensor tells the broker: *“If I disappear without disconnecting, publish `offline`.”* We simulate a power failure.

In [ ]:
recibidos.clear()
s19 = cliente("sensor-019", will=("reefer/019/estado", b"offline", 1, True))
s19.publish("reefer/019/estado", b"online", qos=1, retain=True).wait_for_publish(5)
time.sleep(0.5)
print("Before the outage:", [m["payload"] for m in recibidos if m["topic"] == "reefer/019/estado"])

s19.loop_stop(); s19.socket().close()        # ⚡ corte de corriente: se cierra la connection sin DISCONNECT
time.sleep(1.5)
print("After the outage:", [m["payload"] for m in recibidos if m["topic"] == "reefer/019/estado"])

### 2f · The cost of remaining connected: *keepalive*
MQTT runs over **TCP**, an open connection. To tell the broker you are still alive, an idle client sends `PINGREQ` every *keepalive* seconds. We set it to 3 seconds so we can observe it in class.

In [ ]:
cable.limpiar()
charlatan = cliente("sensor-dormilon", keepalive=3)
time.sleep(7.5)
df = cable.ver("7 seconds without publishing")
charlatan.disconnect(); charlatan.loop_stop()
print(f"\nWith a 60-second keepalive: {86400//60} pings a day = {86400//60} radio wakeups to say 'I am still here'.")

### 🧪 YOUR TURN 2
Publish **your bytes from Turn 1** to `reefer/<your alias>/dato`, choose a QoS level, and inspect the wire. How many MQTT bytes did your reading cost? Which weighs more: the reading or the topic?

In [ ]:
# 🧪 YOUR TURN 2 — QoS 1: acknowledgment from the broker.
cable.limpiar()
MI_QOS = 1
topic = f"reefer/{ALIAS}/dato"
sensor.publish(topic, mis_bytes, qos=MI_QOS).wait_for_publish(5)
time.sleep(0.4)
registro = cable.ver(f"My reading ({len(mis_bytes)} bytes) with QoS {MI_QOS}")
# PUBLISH includes the topic; PUBACK acknowledges delivery to the broker, not the application.
print(f"Reading: {len(mis_bytes)} B; topic: {len(topic.encode())} B. "
      f"Topic is larger: {len(topic.encode()) > len(mis_bytes)}.")
print("The exact MQTT cost is shown above; headers and PUBACK also count.")


> ### ✅ MQTT: what you just observed
> 1. **Everyone talks to the broker**, rather than directly to each other. If it fails, communication fails.
> 2. You must **connect** (CONNECT/CONNACK) and **stay connected** (PINGREQ). Fine on mains power or Wi-Fi; expensive on a battery.
> 3. The **topic** travels in every message and often weighs more than the reading.
> 4. Higher **QoS** means more packets. “Exactly once” reaches only the broker.
> 5. **Retained messages** and **Last Will** address real questions: “What is the latest value?” and “Is the sensor still online?”

---
## Step 3 · Classroom MQTT: everyone on a public broker · slide 50
We now connect to a **public Internet broker** (free for testing, **no password**). Everyone publishes their sensor to `unie-biot-2026/s4/<alias>/…`, while the instructor displays incoming readings. If the public broker does not respond, the cell uses your local broker instead.

In [ ]:
BROKERS_PUBLICOS = ["test.mosquitto.org", "broker.hivemq.com", "broker.emqx.io"]
RAIZ = "unie-biot-2026/s4"

def conectar_publico(nombre):
    for host in BROKERS_PUBLICOS:
        try:
            c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"{nombre}-{random.randint(0, 99999)}")
            c.connect(host, 1883, keepalive=30); c.loop_start(); time.sleep(0.5)
            print("✅ Connected to", host); return c, host
        except Exception as e:
            print("…", host, "did not respond:", type(e).__name__)
    print("⚠️ No public broker available: using local broker (you will only see yourself)")
    return cliente(nombre, puerto=1883), "localhost"

pub, BROKER = conectar_publico(ALIAS)
temp = 4.0 + random.random()
for i in range(12):                                         # one minute: transmit every 5 s
    temp += random.uniform(-0.2, 0.3)
    pub.publish(f"{RAIZ}/{ALIAS}/temp", f"{temp:.2f}", qos=0)
    pub.publish(f"{RAIZ}/{ALIAS}/estado", "online", qos=0, retain=True)
    time.sleep(5)
print("Sent one minute of data to", BROKER)

### 👀 Instructor's cell (also available to anyone else)
Listen to **everything** under `unie-biot-2026/s4/#` for 30 seconds. Nobody asked for a username or password to read other people's data. **That is today's security weakness.**

In [ ]:
aula = {}
def ver_aula(c, u, m):
    partes = m.topic.split("/")
    if len(partes) >= 4:
        aula.setdefault(partes[2], {})[partes[3]] = m.payload.decode(errors="replace")
oyente, _ = conectar_publico("profesor")
oyente.on_message = ver_aula; oyente.subscribe(f"{RAIZ}/#")
time.sleep(30); oyente.loop_stop()
print(f"Devices seen in 30 s: {len(aula)}")
pd.DataFrame(aula).T if aula else print("(nobody has published yet)")

---
## Step 4 · CoAP: the same reading without a connection · slide 63

> ### 💬 CoAP in one sentence
> **It is like ringing the sensor's doorbell and asking a question.** “What is your temperature?” → “4.37”. There is no broker: **the sensor is the server**. Like the web, it uses `GET` to read and `PUT` to change values, but on a smaller scale: a **4-byte** header and separate UDP datagrams instead of a persistent connection.

| | MQTT | CoAP |
|---|---|---|
| Intermediate server? | broker | none |
| Sensor's role? | publishing client | responding **server** |
| Open connection? | yes (TCP) | no (UDP): independent messages |
| Resembles… | WhatsApp | the web (HTTP) |

In [ ]:
diagrama({"your app": (1.5, 2, "#00798c"), "SENSOR\n(CoAP server)": (8.3, 2, "#d1495b")},
         [("your app", "SENSOR\n(CoAP server)", "GET /temp", "#2e4057")],
         "CoAP: query the sensor directly. No broker.", alto=2.6)

We start a CoAP server in your Colab instance (the “sensor”). As with MQTT, we put a **wire** in the middle (this time using UDP) to inspect **every message**.

In [ ]:
#@title Tool · A working CoAP sensor and UDP wire (run it; you need not read it all)
from aiocoap import Message, CON, NON, ACK, GET, PUT, CONTENT, CHANGED, NOT_FOUND, METHOD_NOT_ALLOWED

class SensorCoAP:
    # Minimal CoAP server over UDP. Each resource is a path ("temp") with GET and PUT handlers.
    def __init__(self, puerto=5683):
        self.recursos, self.observadores, self.seq = {}, {}, 2
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); self.sock.bind(("127.0.0.1", puerto))
        threading.Thread(target=self._bucle, daemon=True).start()
    def recurso(self, ruta, get=None, put=None, observable=False):
        self.recursos[ruta] = {"GET": get, "PUT": put, "obs": observable}
    def _bucle(self):
        while True:
            d, quien = self.sock.recvfrom(2048)
            try: m = Message.decode(d)
            except Exception: continue
            ruta = "/".join(m.opt.uri_path); rec = self.recursos.get(ruta)
            r = Message(mtype=ACK if m.mtype == CON else NON)
            if rec is None:                      r.code = NOT_FOUND
            elif m.code == GET and rec["GET"]:
                r.code, r.payload = CONTENT, rec["GET"]()
                if m.opt.observe == 0 and rec["obs"]:
                    self.observadores[(quien, ruta)] = m.token; r.opt.observe = self.seq
            elif m.code == PUT and rec["PUT"]:   r.code, r.payload = CHANGED, rec["PUT"](m.payload)
            else:                                r.code = METHOD_NOT_ALLOWED
            r.mid, r.token = m.mid, m.token
            self.sock.sendto(r.encode(), quien)
    def cambio(self, ruta):                      # avisar a quien observa esa ruta
        self.seq += 1
        for (quien, rr), token in list(self.observadores.items()):
            if rr != ruta: continue
            n = Message(mtype=NON, code=CONTENT, payload=self.recursos[ruta]["GET"]())
            n.opt.observe, n.mid, n.token = self.seq, random.randint(0, 65535), token
            self.sock.sendto(n.encode(), quien)
    def indice(self):                            # /.well-known/core: resource discovery
        return ",".join(f"</{r}>" + (";obs" if v["obs"] else "") for r, v in self.recursos.items()).encode()

class CableUDP:
    TIPO = ["CON", "NON", "ACK", "RST"]
    NOMBRE = {"0.01": "GET", "0.02": "POST", "0.03": "PUT", "0.04": "DELETE", "2.05": "2.05 Content",
              "2.04": "2.04 Changed", "4.04": "4.04 Not Found", "4.05": "4.05 Method Not Allowed"}
    def __init__(self, escucha=5684, destino=("127.0.0.1", 5683)):
        self.registro, self.clientes = [], {}
        self.front = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); self.front.bind(("127.0.0.1", escucha))
        self.back = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); self.destino = destino
        threading.Thread(target=self._ida, daemon=True).start(); threading.Thread(target=self._vuelta, daemon=True).start()
    def _anotar(self, d, direction):
        codigo = f"{d[1]>>5}.{d[1]&31:02d}"
        self.registro.append((direction, self.TIPO[(d[0]>>4)&3], self.NOMBRE.get(codigo, codigo), len(d)))
    def _ida(self):
        while True:
            d, c = self.front.recvfrom(2048); self._anotar(d, "→")
            self.clientes[d[4:4+(d[0]&15)]] = c            # recordamos a who devolver cada token
            self.back.sendto(d, self.destino)
    def _vuelta(self):
        while True:
            d, _ = self.back.recvfrom(2048); self._anotar(d, "←")
            c = self.clientes.get(d[4:4+(d[0]&15)])
            if c: self.front.sendto(d, c)
    def limpiar(self): self.registro.clear()
    def ver(self, titulo=""):
        time.sleep(0.3)
        display(pd.DataFrame(self.registro, columns=["direction", "type", "code", "bytes"]))
        secuencia([(s, f"{t} {c} · {b} B") for s, t, c, b in self.registro], "your app", "sensor", titulo)

def coap(metodo, ruta, payload=b"", puerto=5684, observar=False):
    # CoAP client: send a confirmable request and wait for a response (three retries)
    m = Message(mtype=CON, code=metodo, payload=payload)
    m.opt.uri_path = tuple(p for p in ruta.split("/") if p)
    m.mid, m.token = random.randint(0, 65535), os.urandom(2)
    if observar: m.opt.observe = 0
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); s.settimeout(2)
    for intento in range(3):
        s.sendto(m.encode(), ("127.0.0.1", puerto))
        try:
            r = Message.decode(s.recv(2048))
            if observar: return r, s
            s.close(); return r
        except socket.timeout: continue
    raise TimeoutError("el sensor no contesta")

import os
try:
    sensor_coap
except NameError:
    sensor_coap, cable_udp = SensorCoAP(), CableUDP()
    estado = {"temp": 4.37, "led": b"off"}
    sensor_coap.recurso("temp", get=lambda: struct.pack(">h", round(estado["temp"]*100)), observable=True)
    sensor_coap.recurso("led", get=lambda: estado["led"], put=lambda p: (estado.__setitem__("led", p), b"ok")[1])
    sensor_coap.recurso(".well-known/core", get=sensor_coap.indice)
    def variar():                                        # temperature changes every second
        while True:
            time.sleep(1); estado["temp"] += random.uniform(-0.1, 0.15); sensor_coap.cambio("temp")
    threading.Thread(target=variar, daemon=True).start()
print("📡 Sensor CoAP en coap://127.0.0.1:5683 (resources /temp and /led) · cable UDP en el puerto 5684")

### 4a · Ask the sensor for its temperature
**🎲 Guess:** with MQTT, sending a reading required CONNECT, CONNACK, and PUBLISH. How many messages are required to **get** the temperature with CoAP?

In [ ]:
cable_udp.limpiar()
r = coap(GET, "temp")
print("GET /temp →", r.code, "· payload bytes:", r.payload.hex(" "), "→", struct.unpack(">h", r.payload)[0]/100, "°C")
cable_udp.ver("CoAP: one request, one response. Nothing else.")

Two messages. The reply is an **ACK** that already contains the reading (*piggybacked*). **CON** means *confirmable*: if the ACK does not arrive, the client retries (our `coap` function tries three times).

### 4b · Change something on the sensor (PUT) and ask what it supports

In [ ]:
cable_udp.limpiar()
r = coap(PUT, "led", b"on");         print("PUT /led = on     ", r.code, r.payload)
r = coap(GET, "led");                print("GET /led          ", r.code, r.payload)
r = coap(GET, ".well-known/core");   print("What do you support? ", r.code, r.payload.decode())
r = coap(GET, "no-existe");          print("GET /no-existe    ", r.code)
cable_udp.ver("Turn on an LED, read it, discover resources, and request a missing address")

These codes resemble web status codes: **2.05** corresponds to “200 OK”; **4.04** to “404 Not Found”. CoAP is a compact version of the web.

### 4c · Byte by byte: CoAP versus MQTT versus HTTP
We construct the same request (“give me the temperature”) and **count its bytes**.

In [ ]:
m = Message(mtype=CON, code=GET); m.opt.uri_path = ("temp",); m.mid, m.token = 0x1234, b"\x7a"
crudo = m.encode()
print("CoAP request:", len(crudo), "bytes →", crudo.hex(" "))
print(f"  byte 1 = {crudo[0]:08b}  → version {crudo[0]>>6}, tipo {['CON','NON','ACK','RST'][(crudo[0]>>4)&3]}, token of {crudo[0]&15} byte")
print(f"  byte 2 = {crudo[1]:08b}  → code {crudo[1]>>5}.{crudo[1]&31:02d} (0.01 = GET)")
print(f"  bytes 3-4 = message ID {int.from_bytes(crudo[2:4], 'big'):#06x}  ·  remainder: token + Uri-Path option 'temp'")

resp = Message(code=CONTENT, mtype=ACK, payload=struct.pack(">h", 437)); resp.mid, resp.token = 0x1234, b"\x7a"
http = b"GET /temp HTTP/1.1\r\nHost: sensor\r\n\r\n"
topic = b"reefer/017/temp"; mqtt_pub = bytes([0x30, 2 + len(topic) + 2]) + len(topic).to_bytes(2, "big") + topic + struct.pack(">h", 437)

comparacion = pd.DataFrame([
    {"protocol": "CoAP · GET request", "bytes": len(crudo), "transport": "UDP (8 B header)", "prior connection?": "no"},
    {"protocol": "CoAP · reply with payload", "bytes": len(resp.encode()), "transport": "UDP", "prior connection?": "no"},
    {"protocol": "MQTT · PUBLISH with payload", "bytes": len(mqtt_pub), "transport": "TCP (20 B + handshake)", "prior connection?": "yes: CONNECT/CONNACK"},
    {"protocol": "HTTP · minimal GET request", "bytes": len(http), "transport": "TCP (20 B + handshake)", "prior connection?": "yes"},
]); comparacion

### 4d · Observe: “notify me when it changes”
With **Observe**, the client asks once and requests change notifications; the sensor sends updates itself. This is CoAP's closest equivalent to MQTT, **without an intermediate broker**.

In [ ]:
cable_udp.limpiar()
primera, conexion = coap(GET, "temp", observar=True)
avisos = [struct.unpack(">h", primera.payload)[0]/100]
conexion.settimeout(3)
while len(avisos) < 5:
    avisos.append(struct.unpack(">h", Message.decode(conexion.recv(2048)).payload)[0]/100)
conexion.close(); sensor_coap.observadores.clear()
print("Temperatures received after ONE request:", avisos)
cable_udp.ver("One request; the sensor keeps sending responses")

### 🧪 YOUR TURN 4
Add a `/bateria` resource to the sensor that returns `3.61` V encoded in **1 byte** (hint: `(V-2)*100`) and request it with `GET`. Then try `PUT` on `/temp`: what code does it return, and why?

In [ ]:
# 🧪 YOUR TURN 4 — encode voltage as a one-byte unsigned integer.
sensor_coap.recurso("bateria", get=lambda: bytes([round((3.61 - 2) * 100)]))
r = coap(GET, "bateria")
print("GET /bateria:", r.code, r.payload.hex(), "→", r.payload[0]/100 + 2, "V")
r_put = coap(PUT, "temp", struct.pack(">h", 500))
print("PUT /temp:", r_put.code, "→ 4.05 Method Not Allowed:",
      "the /temp resource supports GET but has no PUT handler.")


> ### ✅ CoAP: what you just observed
> 1. **The sensor is the server**: you query it directly without a broker.
> 2. **One request, one response**: two messages without connecting first or keeping a connection open.
> 3. A **smaller web protocol**: GET, PUT, 2.05 and 4.04 response codes, and resource paths such as `/temp`.
> 4. **Observe** provides change notifications without an intermediate server.
> 5. The trade-off: the sensor must be **reachable**. If it sleeps or a router blocks inbound traffic, it cannot answer.

---
## Step 5 · LoRaWAN: how long each message stays on air · slide 80

> ### 💬 LoRaWAN in one sentence
> **It is like calling out across a field.** Someone several kilometers away may hear you, while a battery lasts years and you pay no SIM fee… but **you can say only a few words a few times a day**, and answers are scarce. The farther away the receiver, **the slower you must speak**.

An important distinction: MQTT and CoAP are **languages** (the message format). LoRaWAN is **the road** (the transport). The network can even deliver LoRaWAN data to your application **through MQTT**:

In [ ]:
diagrama({"sensor\n(5-year battery)": (1.1, 2, "#d1495b"), "gateway A\n(rooftop)": (3.7, 3.1, "#edae49"), "gateway B\n(tower)": (3.7, 0.9, "#edae49"),
          "network\nserver (TTN)": (6.3, 2, "#2e4057"), "your app": (8.9, 2, "#00798c")},
         [("sensor\n(5-year battery)", "gateway A\n(rooftop)", "LoRa radio", "#d1495b"), ("sensor\n(5-year battery)", "gateway B\n(tower)", "LoRa radio", "#d1495b"),
          ("gateway A\n(rooftop)", "network\nserver (TTN)", "Internet", "#666"), ("gateway B\n(tower)", "network\nserver (TTN)", "Internet", "#666"),
          ("network\nserver (TTN)", "your app", "MQTT", "#00798c")],
         "LoRaWAN: the sensor calls out to nearby gateways;\nthe network delivers its data over MQTT", alto=4.4)

In LoRaWAN, the scarce resource is not the number of bytes: it is **radio transmission time** (*airtime*). It depends on **how many bytes** you send and the **SF** (*spreading factor*, 7 to 12). A higher SF reaches farther, but **each symbol takes twice as long** as at the previous SF.

### 5a · Timing the same 2 bytes
We use Semtech's formula (the LoRa chip maker). The network **adds 13 bytes you cannot choose**: header (1), address and counters (7), port (1), and message integrity code, MIC (4).

In [ ]:
CABECERA_LORAWAN = 13          # MHDR 1 + FHDR 7 (without FOpts) + FPort 1 + MIC 4
CUPO_TTN_S = 30                # TTN fair-use limit: 30 s uplink per device per day
MAX_PAYLOAD = {7: 222, 8: 222, 9: 115, 10: 51, 11: 51, 12: 51}   # EU868, application payload bytes (RP002)

def tiempo_en_aire_ms(bytes_app, sf, bw=125_000, cr=1, preambulo=8):
    PL = bytes_app + CABECERA_LORAWAN
    Tsym = 2**sf / bw
    DE = 1 if (sf >= 11 and bw == 125_000) else 0           # low data rate optimization
    n = 8 + max(math.ceil((8*PL - 4*sf + 28 + 16) / (4*(sf - 2*DE))) * (cr + 4), 0)
    return ((preambulo + 4.25) + n) * Tsym * 1000

tabla = pd.DataFrame({f"SF{sf}": {f"{b} B": (round(tiempo_en_aire_ms(b, sf), 1) if b <= MAX_PAYLOAD[sf] else "does not fit")
                                   for b in (1, 2, 4, 12, 24, 51, 115)} for sf in range(7, 13)})
ms = [tiempo_en_aire_ms(2, sf) for sf in range(7, 13)]
plt.figure(figsize=(11, 3.8)); plt.barh([f"SF{sf}" for sf in range(7, 13)], ms, color=["#00798c"]*3 + ["#edae49"] + ["#d1495b"]*2)
for i, v in enumerate(ms): plt.text(v + 15, i, f"{v:.0f} ms", va="center")
plt.gca().invert_yaxis(); plt.xlabel("milliseconds on air"); plt.title("Transmitting “4.37 °C” (2 bytes) at different gateway distances"); plt.show()
print("Airtime per message in milliseconds (EU868, 125 kHz). 'does not fit' = exceeds the payload limit for that SF:"); tabla

### 5b · Listen to it 🔊
The same 2-byte message at SF7 and SF12, **slowed down by a factor of four** so you can hear it. Each “whoosh” is a LoRa symbol (a *chirp*). Turn up the volume.

In [ ]:
def sonido_lora(sf, bytes_app=2, lento=4, fs=16000):
    Tsym = (2**sf / 125_000) * lento
    n = round(tiempo_en_aire_ms(bytes_app, sf) / 1000 / (2**sf / 125_000))      # number of symbols in the whole message
    t = np.arange(int(Tsym*fs)) / fs
    chirp = np.sin(2*np.pi*(300*t + (1500/(2*Tsym))*t**2))                     # sweep from 300 to 1,800 Hz
    return np.tile(chirp, n)

audio = np.concatenate([sonido_lora(7), np.zeros(8000), sonido_lora(12)])
print(f"First SF7: {tiempo_en_aire_ms(2, 7):.0f} actual ms ({4*tiempo_en_aire_ms(2, 7)/1000:.1f} s at slow playback).")
print(f"Then SF12: {tiempo_en_aire_ms(2, 12):.0f} actual ms ({4*tiempo_en_aire_ms(2, 12)/1000:.1f} s at slow playback). Same reading.")
display(Audio(audio, rate=16000))

### 5c · The budget: how often can the sensor speak each day?
Divide TTN's 30-second allowance by the duration of each message. Alongside it is the **regulatory** 1% duty-cycle limit per radio sub-band: 864 seconds a day.

In [ ]:
filas = []
for sf in range(7, 13):
    t = tiempo_en_aire_ms(2, sf)
    filas.append({"SF": sf, "ms per message (2 B)": round(t, 1),
                  "messages/day with 30 s": int(CUPO_TTN_S*1000 // t),
                  "minutes between messages": round(1440 / int(CUPO_TTN_S*1000 // t), 1),
                  "messages/day at 1% duty cycle (per sub-band)": int(864_000 // t)})
presupuesto = pd.DataFrame(filas).set_index("SF"); display(presupuesto)

ax = presupuesto["messages/day with 30 s"].plot.bar(color="#d1495b", rot=0)
for i, v in enumerate(presupuesto["messages/day with 30 s"]): ax.text(i, v + 8, str(v), ha="center")
plt.title("How many messages a day fit within 30 s? (2-byte payload)"); plt.ylabel("messages per day"); plt.show()

**Three observations from the table:**
1. From SF7 to SF12, the allowance drops from **647 to 25 messages per day**. Same sensor and battery; only the gateway distance changed.
2. The **1% regulatory limit** (864 s/day per sub-band) is 29 times more generous than 30 s: **network policy**, rather than regulation, limits this example.
3. **A correction to session 3:** we said *103 messages at SF10*. Inspecting the frame gives that result **only for 0 or 1 payload byte**. With 2 bytes, the result is **90**. We had omitted the 13-byte overhead.

In [ ]:
for b in (0, 1, 2, 12):
    print(f"SF10 with {b:2d} payload bytes → {tiempo_en_aire_ms(b, 10):6.1f} ms → {int(30000 // tiempo_en_aire_ms(b, 10))} messages/day")
print(f"\nWith 2 payload bytes, the header occupies {13/15:.0%} of the frame.")

### 🧪 YOUR TURN 5
Using the bytes from **your** sensor (Turn 1), how many messages per day fit at SF7, SF10, and SF12? How many minutes apart can the messages be?

In [ ]:
# 🧪 YOUR TURN 5 — 2 B of temperature plus modeled LoRaWAN overhead.
MIS_BYTES = len(mis_bytes)
for sf in (7, 10, 12):
    t = tiempo_en_aire_ms(MIS_BYTES, sf)
    n = int(CUPO_TTN_S * 1000 // t)
    print(f"SF{sf}: {t:7.1f} ms/message → {n:4d} messages/day "
          f"→ one every {1440/n:5.1f} min, evenly spaced")


> ### ✅ LoRaWAN: what you just saw (and heard)
> 1. It is **the road**, not the language: readings travel by radio to a gateway and may arrive at your app **through MQTT**.
> 2. **Airtime** is the scarce resource: SF12 takes **25 times** as long as SF7 to say the same thing.
> 3. With 30 seconds a day, a distant sensor (SF12) can send **25 messages a day**; a nearby one (SF7), 647.
> 4. **13 bytes** per message are overhead. Sending one or two payload bytes hardly changes airtime; sending 50 does.
> 5. Downlink is scarcer still: TTN allows **10 messages a day**, including acknowledgments.

---
## Step 6 · What do you let it say? Six strategies within the budget · slide 88
A whole day in a refrigerated container, **measured every minute** (1,440 readings). The setpoint is 4 °C; the contract says **a temperature above 8 °C indicates a failure**. At 14:12 someone leaves the door open.

The sensor **measures** every minute (measurement costs little battery: session 3). The question is what it **transmits**. We test six strategies at **SF10** with a 30-second allowance.

In [ ]:
LIMITE = 8.0
minutos = np.arange(1440)
temp_real = 4.0 + 0.3*np.sin(2*np.pi*minutos/1440) + np.random.normal(0, 0.08, 1440)
for d in (180, 540, 900, 1260):                                     # routine harmless defrost cycles
    temp_real[d:d+20] += 1.4*np.sin(np.linspace(0, np.pi, 20))
t0, dur = 14*60 + 12, 26                                            # open door
subida = 7.6*(1 - np.exp(-np.arange(dur)/9))
temp_real[t0:t0+dur] += subida
temp_real[t0+dur:t0+dur+60] += subida[-1]*np.exp(-np.arange(60)/7)
cruce_real = int(np.argmax(temp_real > LIMITE))
print(f"Actual maximum: {temp_real.max():.2f} °C · minutes above {LIMITE} °C: {(temp_real > LIMITE).sum()} · "
      f"first threshold crossing: {cruce_real//60:02d}:{cruce_real%60:02d}")

plt.plot(minutos/60, temp_real, lw=1, color="#333"); plt.axhline(LIMITE, color="#d1495b", ls="--", label="8 °C limit")
plt.xlabel("hour of day"); plt.ylabel("°C"); plt.title("The full truth (which no receiver gets)"); plt.legend(); plt.xticks(range(0, 25, 2)); plt.show()

In [ ]:
#@title Six strategies (read the comments: each represents an engineering choice)
def cada_minuto(T):              # 1) Send all readings: 1,440 two-byte messages
    return [(m, 2, [T[m]]) for m in range(1440)]

def media_horaria(T):            # 2) The seemingly sensible choice: hourly mean (2 bytes)
    return [(h*60+59, 2, [T[h*60:(h+1)*60].mean()]) for h in range(24)]

def maximo_horario(T):           # 3) Same message count, but hourly MAXIMUM (2 bytes)
    return [(h*60+59, 2, [T[h*60:(h+1)*60].max()]) for h in range(24)]

def resumen_horario(T):          # 4) Hourly minimum, mean, and maximum (6 bytes)
    return [(h*60+59, 6, [T[h*60:(h+1)*60].min(), T[h*60:(h+1)*60].mean(), T[h*60:(h+1)*60].max()]) for h in range(24)]

def lote_5min(T):                # 5) Five-minute series: 12 readings per hour, one byte each (0.25 °C steps)
    return [(h*60+59, 12, [T[h*60+k*5] for k in range(12)]) for h in range(24)]

def por_excepcion(T, histeresis=7.5, latido_h=6):   # 6) Transmit only on events plus a heartbeat every 6 h
    msgs, alarma = [], False
    for m in range(1440):
        if not alarma and T[m] > LIMITE:  msgs.append((m, 3, [T[m]])); alarma = True     # alert!
        elif alarma and T[m] < histeresis: msgs.append((m, 3, [T[m]])); alarma = False   # back to normal
        if m % (latido_h*60) == latido_h*60 - 1:
            v = T[m-latido_h*60+1:m+1]; msgs.append((m, 6, [v.min(), v.mean(), v.max()]))
    return msgs

ESTRATEGIAS = {"1 · Every minute": cada_minuto, "2 · Hourly mean": media_horaria, "3 · Hourly maximum": maximo_horario,
               "4 · Hourly min/mean/max": resumen_horario, "5 · Five-minute batch": lote_5min, "6 · Event-based": por_excepcion}

def evaluar(estrategia, T, sf=10, cupo_s=CUPO_TTN_S):
    msgs = estrategia(T)
    segundos = sum(tiempo_en_aire_ms(b, sf) for _, b, _ in msgs) / 1000
    cabe_tamano = all(b <= MAX_PAYLOAD[sf] for _, b, _ in msgs)
    avisos = [m for m, _, vals in msgs if max(vals) > LIMITE]
    return {"messages/day": len(msgs), "bytes/message": max(b for _, b, _ in msgs),
            "seconds on air": round(segundos, 1), "fits in 30 s?": "✅" if segundos <= cupo_s and cabe_tamano else "❌",
            "failure detected?": "✅" if avisos else "❌ hidden",
            "alert delay (min)": (avisos[0] - cruce_real) if avisos else "—"}

In [ ]:
resultado = pd.DataFrame({n: evaluar(f, temp_real, sf=10) for n, f in ESTRATEGIAS.items()}).T
resultado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(minutos/60, temp_real, lw=0.8, color="#bbb", label="full readings")
for nombre, f, color in [("2 · Hourly mean", media_horaria, "#7a9cc6"), ("3 · Hourly maximum", maximo_horario, "#d1495b")]:
    msgs = f(temp_real); ax.step([(m-59)/60 for m, _, _ in msgs], [v[0] for _, _, v in msgs], where="post", lw=2.2, color=color, label=nombre)
ax.axhline(LIMITE, color="k", ls="--", lw=1); ax.set_xlim(11, 18); ax.set_xlabel("hora"); ax.set_ylabel("°C")
ax.set_title("One value per hour. Same 24 messages: the mean hides the failure, the maximum preserves it"); ax.legend(); plt.show()

**What the table tells us** is the answer to this session:
- **Sending everything exceeds the budget.** Even if you wanted to do it.
- **An hourly mean fits but hides the failure.** Many people choose it because it sounds statistically sensible.
- **An hourly maximum** costs exactly the same but **preserves the failure**. That is an engineering decision, not an extra fee.
- **Event-based reporting** consumes the least airtime and **alerts earliest**: the sensor transmits decisions instead of all readings.

### 🧪 YOUR TURN 6 · Design your strategy
Write a function returning a list of `(minute, bytes, [values])`. It must **fit within 30 s at SF10** and **detect the failure**. Bonus: alert within five minutes.

In [ ]:
# 🧪 YOUR TURN 6 — immediate threshold alert, recovery with hysteresis,
# and a heartbeat every 6 h. Event messages include status (1 B) and temperature
# (2 B); heartbeats carry temperature (2 B). Minute 0 is the first sample.
def mi_estrategia(T):
    msgs, en_alarma = [], False
    for minuto, valor in enumerate(T):
        if not en_alarma and valor > LIMITE:
            msgs.append((minuto, 3, [float(valor)]))  # status=alert
            en_alarma = True
        elif en_alarma and valor < LIMITE - 0.5:
            msgs.append((minuto, 3, [float(valor)]))  # status=recovered
            en_alarma = False
        if (minuto + 1) % 360 == 0:
            msgs.append((minuto, 2, [float(valor)]))  # still alive
    return msgs

display(pd.DataFrame({"my strategy": evaluar(mi_estrategia, temp_real, sf=10)}).T)
print("The alert is sent at the first sample above 8 °C; modeled delay: 0 min.")


---
## Step 7 · The weak point: when the gateway that heard you disappears · slide 92
You do not choose the SF alone: **ADR** (*Adaptive Data Rate*) lets the network adjust it according to reception. If the nearby rooftop *gateway* goes offline (someone else paid for it), your sensor has to reach a more distant one… at **SF12**.

The same strategies on the same day at **three distances**.

In [ ]:
filas = {}
for sf in (7, 10, 12):
    for n, f in ESTRATEGIAS.items():
        r = evaluar(f, temp_real, sf=sf)
        filas.setdefault(n, {})[f"SF{sf}"] = f"{r['fits in 30 s?']} {r['seconds on air']} s"
pd.DataFrame(filas).T

**Your budget depends on somebody else's rooftop.** The batch, the most detailed strategy at SF10 (the complete hourly curve), **exceeds the allowance at SF12**. So does the min/mean/max summary. The hourly maximum fits (barely), while event-based reporting has ample room. Good design chooses a strategy that **survives the worst SF**, not just the SF shown in a demonstration.

### 🧪 YOUR TURN 7
Does your Turn 6 strategy survive SF12? If not, fix it.

In [ ]:
# 🧪 YOUR TURN 7 — also test the worst expected SF.
comparacion_sf = pd.DataFrame({f"SF{sf}": evaluar(mi_estrategia, temp_real, sf=sf)
                               for sf in (7, 10, 12)}).T
display(comparacion_sf)
assert comparacion_sf.loc["SF12", "fits in 30 s?"] == "✅"
assert comparacion_sf.loc["SF12", "failure detected?"] == "✅"
print("It survives SF12 for this simulated day; with many alerts,")
print("recalculate airtime and prioritize events over heartbeats.")


---
## 🎯 Quick review · Which would you choose?
Five practical cases. Enter `"MQTT"`, `"CoAP"`, or `"LoRaWAN"` for each and run the cell. (If two choices seem plausible, choose one and defend it aloud: sometimes both are valid.)

In [ ]:
mis_respuestas = {
    "1. Supermarket cold room with Wi-Fi and mains power; three displays need the data": "MQTT",
    "2. Battery-powered soil moisture sensor in an olive grove 4 km from town": "LoRaWAN",
    "3. Basement water meter with carrier NB-IoT modem and ten-year battery": "CoAP",
    "4. Control panel that must OPEN a valve immediately, several times a day": "MQTT",
    "5. Waste containers across a city reporting when they are full": "LoRaWAN",
}
SOLUCION = {1: ("MQTT", "mains power and networking; the broker serves multiple subscribers"),
            2: ("LoRaWAN", "kilometers away, battery power, small payloads: long-range radio"),
            3: ("CoAP", "carrier IP network, few bytes, no persistent connection (often LwM2M over CoAP)"),
            4: ("MQTT", "commands must arrive often; LoRaWAN allows only 10 downlinks a day and a Class A sensor listens only after transmitting"),
            5: ("LoRaWAN", "thousands of battery devices sending little data; NB-IoT + CoAP also works if the city pays a carrier")}
aciertos = 0
for i, (caso, resp) in enumerate(mis_respuestas.items(), 1):
    ok = resp.strip().lower() == SOLUCION[i][0].lower(); aciertos += ok
    print(f"{'✅' if ok else '❌'} {caso}\n    your answer: {resp} · suggested: {SOLUCION[i][0]} — {SOLUCION[i][1]}\n")
print(f"{aciertos}/5")


---
## Step 8 · Column 6 of your worksheet (ACT 1) · slide 104
Fill this in and **copy the result into your worksheet**. This is the column left open in session 3: *how your sensor communicates*.

In [ ]:
# 🧪 YOUR TURN 8 — same reading and strategy as in previous turns.
mensajes = mi_estrategia(temp_real)
ficha = {
    "Sensor (from your worksheet)": "SHT40, refrigerated-container temperature (illustrative scenario)",
    "Network / protocol": "LoRaWAN (TTN); gateway backhaul over MQTT",
    "Why this choice": "no guaranteed Wi-Fi, battery power, short messages; verify coverage",
    "Bytes per message": "2 B per heartbeat; 3 B per alarm or recovery",
    "Messages per day": len(mensajes),
    "Design SF (worst case)": 12,
    "What it transmits (strategy)": "alarm above 8 °C, recovery below 7.5 °C, and heartbeat every 6 h",
    "What it does not transmit (accepted trade-off)": "intermediate readings, humidity, and exact measurement time; the gateway timestamps arrival",
}
tiempo = sum(tiempo_en_aire_ms(b, 12) for _, b, _ in mensajes) / 1000
ficha["Seconds on air per day"] = f"{tiempo:.1f} s de {CUPO_TTN_S:g} → {'✅ fits' if tiempo <= CUPO_TTN_S else '❌ DOES NOT FIT'}"
for k, v in ficha.items():
    print(f"{k:32s} {v}")
print("Calculation for the simulated day; allow for more events in production.")


---
### Homework
1. Copy column 6 neatly into your ACT 1 worksheet, with **one sentence** explaining what you omit and why.
2. Rerun Step 6 with `LIMITE` set to 6 °C. Which strategy starts producing **false alarms** during defrost cycles?

**Looking ahead to session 5:** now you know what you let it say. **Whom do you tell, and how much do they charge to listen?**

In [ ]:
#@title Cleanup (optional, when finished)
for c in [globals().get(n) for n in ("panel", "sensor", "pub")]:
    try: c.disconnect(); c.loop_stop()
    except Exception: pass
print("Clients disconnected.")